# 02 – Model Experiments
Task: per (GPS point, candidate arc) row, predict y = 1 if the candidate is the true arc.
- **Train:** 80 training trips, reliable labels only. **Compare:** 10 student_test trips.
- **Cross-validation:** 5-fold **GroupKFold by trip**, so points of one trip never appear in both
  training and validation folds (neighbouring GPS points are nearly identical, so splitting a
  trip would leak). For speed, CV uses a random 30% of training GPS points (whole points, so each
  point's candidate set stays together); final models are fitted on all training rows.
- **Metrics:** CV Macro F1; on student_test: Top-1 accuracy (per GPS point, pick the candidate with
  the highest probability), Macro F1, precision, recall, log loss, route continuity, training
  and inference time.
- **Primary selection metric: student_test Top-1 accuracy**, because it measures what the system
  outputs (one arc per GPS point). Macro F1 is used in CV because classes are imbalanced (15.3% positive).

In [ ]:
import os, time, zipfile, joblib
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from google.colab import drive
from sklearn.base import clone
from sklearn.model_selection import GroupKFold
from sklearn.metrics import f1_score, precision_score, recall_score, log_loss, confusion_matrix

drive.mount("/content/drive")
OUT = "/content/drive/MyDrive/pathsnap"
os.makedirs(f"{OUT}/models", exist_ok=True)

# Raw data (needed only for arcs, for route continuity)
with zipfile.ZipFile(f"{OUT}/Student folder.zip") as z:
    z.extractall("/content/data")
DATA_ROOT = "/content/data/Student folder"

SEED = 42
FEATURES = ["dist_m", "dist_rank", "dist_minus_min", "heading_diff", "proj_frac",
            "at_endpoint", "arc_length_m", "n_cands", "is_twoway"]

final = pd.read_parquet(f"{OUT}/features_R50_K10.parquet")
points = pd.read_csv(f"{OUT}/point_labels.csv", dtype={"trip": str})
final["trip"] = final.trip.astype(str)

train = final[(final.partition == "train") & final.label_ok].reset_index(drop=True)
test = final[(final.partition == "student_test") & final.label_ok].reset_index(drop=True)
N_TEST_POINTS = int(points[(points.partition == "student_test") & points.label_ok].shape[0])

# CV subset: 30% of training GPS points (whole points, so candidate sets stay intact).
# Integer id per GPS point + hash lookup (fast).
rng = np.random.default_rng(SEED)
point_key = train.groupby(["trip", "point"]).ngroup()
n_pts = point_key.max() + 1
chosen = rng.choice(n_pts, size=int(0.3 * n_pts), replace=False)
cv_df = train[point_key.isin(chosen)].reset_index(drop=True)

print(f"train rows {len(train):,} | CV rows {len(cv_df):,} | test rows {len(test):,} | test points {N_TEST_POINTS:,}")
print(f"positive rate train {train.y.mean():.3f}, test {test.y.mean():.3f}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
train rows 1,200,125 | CV rows 359,685 | test rows 131,570 | test points 19,133
positive rate train 0.153, test 0.145


In [ ]:
gkf = GroupKFold(n_splits=5)

def cv_macro_f1(model):
    scores = []
    for tr_idx, va_idx in gkf.split(cv_df, cv_df.y, groups=cv_df.trip):
        m = clone(model).fit(cv_df.loc[tr_idx, FEATURES], cv_df.loc[tr_idx, "y"])
        scores.append(f1_score(cv_df.loc[va_idx, "y"], m.predict(cv_df.loc[va_idx, FEATURES]), average="macro"))
    return np.mean(scores), np.std(scores)

def top1_choices(df, proba):
    """Per GPS point, the candidate with the highest predicted probability."""
    d = df[["trip", "point", "arc", "segment", "y"]].copy()
    d["p"] = proba
    return d.sort_values("p", ascending=False).groupby(["trip", "point"]).head(1).sort_values(["trip", "point"])

TEST_ARCS = {t: pd.read_csv(f"{DATA_ROOT}/student_test/{t}/{t}.arcs", sep=r"\s+", header=None, names=["u", "v"])
             for t in test.trip.unique()}

def route_continuity(best):
    """Share of consecutive GPS points (same segment) whose predicted arcs are the same or connected."""
    ok_all = []
    for t, b in best.groupby("trip"):
        U, V = TEST_ARCS[t].u.values, TEST_ARCS[t].v.values
        a = b.arc.values
        consecutive = (np.diff(b.point.values) == 1) & (np.diff(b.segment.values) == 0)
        ok = (a[:-1] == a[1:]) | (V[a[:-1]] == U[a[1:]])
        ok_all.append(ok[consecutive])
    return np.concatenate(ok_all).mean()

results, probas = [], {}

def run_model(name, family, config, model):
    print(f"--- {name}"); t_start = time.time()
    cv_mean, cv_std = cv_macro_f1(model)
    t0 = time.time(); m = clone(model).fit(train[FEATURES], train.y); train_time = time.time() - t0
    t0 = time.time(); proba = m.predict_proba(test[FEATURES])[:, 1]; inf_time = time.time() - t0
    pred = (proba >= 0.5).astype(int)
    best = top1_choices(test, proba)
    res = {"model": name, "family": family, "config": config,
           "cv_macro_f1": round(cv_mean, 4), "cv_std": round(cv_std, 4),
           "test_top1_acc": round(best.y.sum() / N_TEST_POINTS, 4),
           "test_macro_f1": round(f1_score(test.y, pred, average="macro"), 4),
           "test_precision": round(precision_score(test.y, pred, average="macro"), 4),
           "test_recall": round(recall_score(test.y, pred, average="macro"), 4),
           "test_log_loss": round(log_loss(test.y, np.clip(proba, 1e-6, 1 - 1e-6)), 4),
           "route_continuity": round(route_continuity(best), 4),
           "train_time_s": round(train_time, 1),
           "inference_ms_per_point": round(1000 * inf_time / N_TEST_POINTS, 4)}
    results.append(res); probas[name] = proba
    pd.DataFrame(results).to_csv(f"{OUT}/model_comparison.csv", index=False)
    joblib.dump(m, f"{OUT}/models/{name}.joblib")
    print({k: v for k, v in res.items() if k not in ("family", "config")}, f"| total {time.time()-t_start:.0f}s")
    return m

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier

run_model("LogisticRegression", "Linear", "StandardScaler + LR, C=1, class_weight=balanced",
          Pipeline([("sc", StandardScaler()),
                    ("m", LogisticRegression(C=1.0, class_weight="balanced", max_iter=1000))]))

run_model("GaussianNB", "Probabilistic", "default (var_smoothing=1e-9)", GaussianNB())

run_model("DecisionTree", "Tree", "max_depth=12, min_samples_leaf=50, class_weight=balanced",
          DecisionTreeClassifier(max_depth=12, min_samples_leaf=50, class_weight="balanced", random_state=SEED))

pd.DataFrame(results)

--- LogisticRegression
{'model': 'LogisticRegression', 'cv_macro_f1': np.float64(0.8705), 'cv_std': np.float64(0.018), 'test_top1_acc': np.float64(0.8509), 'test_macro_f1': 0.8698, 'test_precision': 0.833, 'test_recall': 0.9272, 'test_log_loss': 0.2047, 'route_continuity': np.float64(0.9196), 'train_time_s': 5.1, 'inference_ms_per_point': 0.001} | total 16s
--- GaussianNB
{'model': 'GaussianNB', 'cv_macro_f1': np.float64(0.7986), 'cv_std': np.float64(0.0149), 'test_top1_acc': np.float64(0.8284), 'test_macro_f1': 0.7979, 'test_precision': 0.7592, 'test_recall': 0.8928, 'test_log_loss': 0.5774, 'route_continuity': np.float64(0.9065), 'train_time_s': 0.4, 'inference_ms_per_point': 0.002} | total 2s
--- DecisionTree
{'model': 'DecisionTree', 'cv_macro_f1': np.float64(0.877), 'cv_std': np.float64(0.0183), 'test_top1_acc': np.float64(0.8502), 'test_macro_f1': 0.8824, 'test_precision': 0.849, 'test_recall': 0.9304, 'test_log_loss': 0.1853, 'route_continuity': np.float64(0.9142), 'train_time_s

,model,family,config,cv_macro_f1,cv_std,test_top1_acc,test_macro_f1,test_precision,test_recall,test_log_loss,route_continuity,train_time_s,inference_ms_per_point
0,LogisticRegression,Linear,"StandardScaler + LR, C=1, class_weight=balanced",0.8705,0.0180,0.8509,0.8698,0.8330,0.9272,0.2047,0.9196,5.1,0.0010
1,GaussianNB,Probabilistic,default (var_smoothing=1e-9),0.7986,0.0149,0.8284,0.7979,0.7592,0.8928,0.5774,0.9065,0.4,0.0020
2,DecisionTree,Tree,"max_depth=12, min_samples_leaf=50, class_weigh...",0.8770,0.0183,0.8502,0.8824,0.8490,0.9304,0.1853,0.9142,20.3,0.0015


In [ ]:
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.neural_network import MLPClassifier

run_model("RandomForest", "Ensemble (bagging)",
          "100 trees, max_depth=16, min_samples_leaf=20, max_samples=0.3, class_weight=balanced",
          RandomForestClassifier(n_estimators=100, max_depth=16, min_samples_leaf=20, max_samples=0.3,
                                 class_weight="balanced", n_jobs=-1, random_state=SEED))

run_model("HistGradientBoosting", "Boosting",
          "max_iter=300, learning_rate=0.1, max_leaf_nodes=31, early_stopping, class_weight=balanced",
          HistGradientBoostingClassifier(max_iter=300, learning_rate=0.1, max_leaf_nodes=31,
                                         early_stopping=True, class_weight="balanced", random_state=SEED))

run_model("KNN", "Instance-based", "StandardScaler + k=25, distance-weighted",
          Pipeline([("sc", StandardScaler()),
                    ("m", KNeighborsClassifier(n_neighbors=25, weights="distance", n_jobs=-1))]))

mlp = run_model("MLP", "Neural network",
                "StandardScaler + hidden (64, 32), ReLU, Adam, batch 2048, early stopping",
                Pipeline([("sc", StandardScaler()),
                          ("m", MLPClassifier(hidden_layer_sizes=(64, 32), activation="relu", solver="adam",
                                              batch_size=2048, max_iter=50, early_stopping=True,
                                              validation_fraction=0.1, random_state=SEED))]))

pd.DataFrame(results)

--- RandomForest
{'model': 'RandomForest', 'cv_macro_f1': np.float64(0.8908), 'cv_std': np.float64(0.0193), 'test_top1_acc': np.float64(0.8493), 'test_macro_f1': 0.8926, 'test_precision': 0.862, 'test_recall': 0.9343, 'test_log_loss': 0.1613, 'route_continuity': np.float64(0.9152), 'train_time_s': 182.5, 'inference_ms_per_point': 0.0749} | total 351s
--- HistGradientBoosting
{'model': 'HistGradientBoosting', 'cv_macro_f1': np.float64(0.8865), 'cv_std': np.float64(0.0194), 'test_top1_acc': np.float64(0.8506), 'test_macro_f1': 0.8847, 'test_precision': 0.8504, 'test_recall': 0.9342, 'test_log_loss': 0.1748, 'route_continuity': np.float64(0.9139), 'train_time_s': 23.8, 'inference_ms_per_point': 0.0725} | total 94s
--- KNN
{'model': 'KNN', 'cv_macro_f1': np.float64(0.9012), 'cv_std': np.float64(0.0168), 'test_top1_acc': np.float64(0.8492), 'test_macro_f1': 0.9084, 'test_precision': 0.9131, 'test_recall': 0.9039, 'test_log_loss': 0.1642, 'route_continuity': np.float64(0.9136), 'train_time_s

/usr/local/lib/python3.13/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (50) reached and the optimization hasn't converged yet.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (50) reached and the optimization hasn't converged yet.
  warnings.warn(


{'model': 'MLP', 'cv_macro_f1': np.float64(0.9028), 'cv_std': np.float64(0.0173), 'test_top1_acc': np.float64(0.85), 'test_macro_f1': 0.9095, 'test_precision': 0.9157, 'test_recall': 0.9036, 'test_log_loss': 0.117, 'route_continuity': np.float64(0.916), 'train_time_s': 138.8, 'inference_ms_per_point': 0.0059} | total 414s


,model,family,config,cv_macro_f1,cv_std,test_top1_acc,test_macro_f1,test_precision,test_recall,test_log_loss,route_continuity,train_time_s,inference_ms_per_point
0,LogisticRegression,Linear,"StandardScaler + LR, C=1, class_weight=balanced",0.8705,0.0180,0.8509,0.8698,0.8330,0.9272,0.2047,0.9196,5.1,0.0010
1,GaussianNB,Probabilistic,default (var_smoothing=1e-9),0.7986,0.0149,0.8284,0.7979,0.7592,0.8928,0.5774,0.9065,0.4,0.0020
2,DecisionTree,Tree,"max_depth=12, min_samples_leaf=50, class_weigh...",0.8770,0.0183,0.8502,0.8824,0.8490,0.9304,0.1853,0.9142,20.3,0.0015
3,RandomForest,Ensemble (bagging),"100 trees, max_depth=16, min_samples_leaf=20, ...",0.8908,0.0193,0.8493,0.8926,0.8620,0.9343,0.1613,0.9152,182.5,0.0749
4,HistGradientBoosting,Boosting,"max_iter=300, learning_rate=0.1, max_leaf_node...",0.8865,0.0194,0.8506,0.8847,0.8504,0.9342,0.1748,0.9139,23.8,0.0725
5,KNN,Instance-based,"StandardScaler + k=25, distance-weighted",0.9012,0.0168,0.8492,0.9084,0.9131,0.9039,0.1642,0.9136,5.1,2.0289
6,MLP,Neural network,"StandardScaler + hidden (64, 32), ReLU, Adam, ...",0.9028,0.0173,0.8500,0.9095,0.9157,0.9036,0.1170,0.9160,138.8,0.0059


In [ ]:
best = top1_choices(test, probas["MLP"])
true_arc = test[test.y == 1].groupby(["trip", "point"]).arc.first().rename("true_arc")
b = best.join(true_arc, on=["trip", "point"])
wrong = b[b.y == 0]

counts = {"twin (opposite direction)": 0, "adjacent (shares a node)": 0, "other road": 0, "true arc not in candidates": 0}
for t, g in wrong.groupby("trip"):
    U, V = TEST_ARCS[t].u.values, TEST_ARCS[t].v.values
    for p_arc, t_arc in zip(g.arc.values, g.true_arc.values):
        if np.isnan(t_arc): counts["true arc not in candidates"] += 1; continue
        t_arc = int(t_arc)
        if U[p_arc] == V[t_arc] and V[p_arc] == U[t_arc]: counts["twin (opposite direction)"] += 1
        elif {U[p_arc], V[p_arc]} & {U[t_arc], V[t_arc]}: counts["adjacent (shares a node)"] += 1
        else: counts["other road"] += 1
counts["no candidates at all"] = N_TEST_POINTS - len(best)
err = pd.Series(counts); print((100 * err / N_TEST_POINTS).round(2).astype(str) + " % of test points")
err.to_csv(f"{OUT}/error_analysis_mlp.csv")

twin (opposite direction)     6.58 % of test points
adjacent (shares a node)      5.26 % of test points
other road                    3.04 % of test points
true arc not in candidates    0.12 % of test points
no candidates at all           0.0 % of test points
dtype: object


In [ ]:
tw = []
for t, g in wrong.groupby("trip"):
    U, V = TEST_ARCS[t].u.values, TEST_ARCS[t].v.values
    ok = ~np.isnan(g.true_arc.values)
    pa, ta = g.arc.values[ok], g.true_arc.values[ok].astype(int)
    is_twin = (U[pa] == V[ta]) & (V[pa] == U[ta])
    tw.append(g[ok][is_twin][["trip", "point"]])
tw = pd.concat(tw)
pt_info = test.groupby(["trip", "point"])[["speed_mps", "heading_valid"]].first()
twin_info = pt_info.loc[list(zip(tw.trip, tw.point))]
print("Twin errors   : median speed %.1f m/s, heading_valid %.1f%%" % (twin_info.speed_mps.median(), 100 * twin_info.heading_valid.mean()))
print("All test points: median speed %.1f m/s, heading_valid %.1f%%" % (pt_info.speed_mps.median(), 100 * pt_info.heading_valid.mean()))

Twin errors   : median speed 7.0 m/s, heading_valid 93.9%
All test points: median speed 14.1 m/s, heading_valid 99.0%


In [ ]:
# Reload with context features
final = pd.read_parquet(f"{OUT}/features_R50_K10_ctx.parquet")
final["trip"] = final.trip.astype(str)

BASE_FEATURES = ["dist_m", "dist_rank", "dist_minus_min", "heading_diff", "proj_frac",
                 "at_endpoint", "arc_length_m", "n_cands", "is_twoway"]
CTX_FEATURES = ["dist_prev_m", "dist_next_m", "ctx_dist_m", "ctx_minus_min", "ctx_rank", "heading_diff_w3"]
FEATURES = BASE_FEATURES + CTX_FEATURES          # run_model uses this global

train = final[(final.partition == "train") & final.label_ok].reset_index(drop=True)
test = final[(final.partition == "student_test") & final.label_ok].reset_index(drop=True)

rng = np.random.default_rng(SEED)
point_key = train.groupby(["trip", "point"]).ngroup()
n_pts = point_key.max() + 1
chosen = rng.choice(n_pts, size=int(0.3 * n_pts), replace=False)
cv_df = train[point_key.isin(chosen)].reset_index(drop=True)
print(f"{len(FEATURES)} features | train {len(train):,} | CV {len(cv_df):,} | test {len(test):,}")

run_model("HistGradientBoosting_ctx", "Boosting", "same as before + 6 context features",
          HistGradientBoostingClassifier(max_iter=300, learning_rate=0.1, max_leaf_nodes=31,
                                         early_stopping=True, class_weight="balanced", random_state=SEED))

run_model("LogisticRegression_ctx", "Linear", "same as before + 6 context features",
          Pipeline([("sc", StandardScaler()),
                    ("m", LogisticRegression(C=1.0, class_weight="balanced", max_iter=1000))]))

mlp_ctx = run_model("MLP_ctx", "Neural network", "hidden (64, 32), Adam, batch 2048, early stopping, max_iter=100",
                    Pipeline([("sc", StandardScaler()),
                              ("m", MLPClassifier(hidden_layer_sizes=(64, 32), activation="relu", solver="adam",
                                                  batch_size=2048, max_iter=100, early_stopping=True,
                                                  validation_fraction=0.1, random_state=SEED))]))

pd.DataFrame(results)[["model", "cv_macro_f1", "test_top1_acc", "test_macro_f1", "test_log_loss", "route_continuity"]]

15 features | train 1,200,125 | CV 359,685 | test 131,570
--- HistGradientBoosting_ctx
{'model': 'HistGradientBoosting_ctx', 'cv_macro_f1': np.float64(0.9256), 'cv_std': np.float64(0.0134), 'test_top1_acc': np.float64(0.8898), 'test_macro_f1': 0.9152, 'test_precision': 0.8847, 'test_recall': 0.9554, 'test_log_loss': 0.1213, 'route_continuity': np.float64(0.9543), 'train_time_s': 25.6, 'inference_ms_per_point': 0.0636} | total 106s
--- LogisticRegression_ctx
{'model': 'LogisticRegression_ctx', 'cv_macro_f1': np.float64(0.9049), 'cv_std': np.float64(0.0089), 'test_top1_acc': np.float64(0.881), 'test_macro_f1': 0.8961, 'test_precision': 0.862, 'test_recall': 0.9445, 'test_log_loss': 0.1573, 'route_continuity': np.float64(0.9508), 'train_time_s': 5.8, 'inference_ms_per_point': 0.0013} | total 15s
--- MLP_ctx
{'model': 'MLP_ctx', 'cv_macro_f1': np.float64(0.935), 'cv_std': np.float64(0.0098), 'test_top1_acc': np.float64(0.8922), 'test_macro_f1': 0.9337, 'test_precision': 0.9376, 'test_recal

,model,cv_macro_f1,test_top1_acc,test_macro_f1,test_log_loss,route_continuity
0,LogisticRegression,0.8705,0.8509,0.8698,0.2047,0.9196
1,GaussianNB,0.7986,0.8284,0.7979,0.5774,0.9065
2,DecisionTree,0.8770,0.8502,0.8824,0.1853,0.9142
3,RandomForest,0.8908,0.8493,0.8926,0.1613,0.9152
4,HistGradientBoosting,0.8865,0.8506,0.8847,0.1748,0.9139
5,KNN,0.9012,0.8492,0.9084,0.1642,0.9136
6,MLP,0.9028,0.8500,0.9095,0.1170,0.9160
7,HistGradientBoosting_ctx,0.9256,0.8898,0.9152,0.1213,0.9543
8,LogisticRegression_ctx,0.9049,0.8810,0.8961,0.1573,0.9508
9,MLP_ctx,0.9350,0.8922,0.9337,0.0810,0.9601


In [ ]:
!pip install -q optuna
import optuna

# Tuning subset: half of the CV points (whole points), 3-fold grouped CV
pk = cv_df.groupby(["trip", "point"]).ngroup()
sel = np.random.default_rng(SEED).choice(pk.max() + 1, size=(pk.max() + 1) // 2, replace=False)
tune_df = cv_df[pk.isin(sel)].reset_index(drop=True)
gkf3 = GroupKFold(n_splits=3)
print(f"tuning rows: {len(tune_df):,}")

def fold_top1(df, proba):
    d = df[["trip", "point", "y"]].copy(); d["p"] = proba
    return d.sort_values("p", ascending=False).groupby(["trip", "point"]).head(1).y.mean()

def make_mlp(params):
    layers = (params["units1"],) if params["units2"] == 0 else (params["units1"], params["units2"])
    return Pipeline([("sc", StandardScaler()),
                     ("m", MLPClassifier(hidden_layer_sizes=layers, activation=params["activation"],
                                         alpha=params["alpha"], learning_rate_init=params["lr"],
                                         batch_size=params["batch_size"], solver="adam", max_iter=100,
                                         early_stopping=True, validation_fraction=0.1, random_state=SEED))])

def objective(trial):
    params = {"units1": trial.suggest_categorical("units1", [32, 64, 128]),
              "units2": trial.suggest_categorical("units2", [0, 16, 32, 64]),
              "activation": trial.suggest_categorical("activation", ["relu", "tanh"]),
              "alpha": trial.suggest_float("alpha", 1e-6, 1e-2, log=True),
              "lr": trial.suggest_float("lr", 1e-4, 1e-2, log=True),
              "batch_size": trial.suggest_categorical("batch_size", [1024, 2048, 4096])}
    scores, losses = [], []
    for k, (tr, va) in enumerate(gkf3.split(tune_df, tune_df.y, groups=tune_df.trip)):
        m = make_mlp(params).fit(tune_df.loc[tr, FEATURES], tune_df.loc[tr, "y"])
        p = m.predict_proba(tune_df.loc[va, FEATURES])[:, 1]
        scores.append(fold_top1(tune_df.loc[va], p))
        losses.append(log_loss(tune_df.loc[va, "y"], np.clip(p, 1e-6, 1 - 1e-6)))
        trial.report(np.mean(scores), k)
        if trial.should_prune():
            raise optuna.TrialPruned()
    trial.set_user_attr("cv_log_loss", float(np.mean(losses)))
    return float(np.mean(scores))

study = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=SEED),
                            pruner=optuna.pruners.MedianPruner(n_startup_trials=3))
# Trial 0 = the untuned configuration, for a fair tuned-vs-untuned comparison
study.enqueue_trial({"units1": 64, "units2": 32, "activation": "relu", "alpha": 1e-4, "lr": 1e-3, "batch_size": 2048})
t0 = time.time()
study.optimize(objective, n_trials=20, timeout=1500)       # stops after 20 trials or 25 minutes
print(f"optimisation time: {time.time() - t0:.0f} s, trials: {len(study.trials)}")

trials = study.trials_dataframe()
trials.to_csv(f"{OUT}/optuna_mlp_trials.csv", index=False)
print("untuned (trial 0) CV Top-1:", round(study.trials[0].value, 4))
print("best CV Top-1:", round(study.best_value, 4), "| best params:", study.best_params)

[I 2026-11-18 07:00:12,509] A new study created in memory with name: no-name-63427821-0234-41b1-9857-c7a2f91cbd0c


tuning rows: 179,782


[I 2026-11-18 07:01:33,511] Trial 0 finished with value: 0.8896225547314872 and parameters: {'units1': 64, 'units2': 32, 'activation': 'relu', 'alpha': 0.0001, 'lr': 0.001, 'batch_size': 2048}. Best is trial 0 with value: 0.8896225547314872.
[I 2026-11-18 07:03:16,574] Trial 1 finished with value: 0.8869836563077759 and parameters: {'units1': 64, 'units2': 0, 'activation': 'relu', 'alpha': 0.0006796578090758161, 'lr': 0.00010994335574766199, 'batch_size': 1024}. Best is trial 0 with value: 0.8896225547314872.
[I 2026-11-18 07:06:22,838] Trial 2 finished with value: 0.8921398317992478 and parameters: {'units1': 128, 'units2': 64, 'activation': 'tanh', 'alpha': 2.9204338471814107e-05, 'lr': 0.000816845589476017, 'batch_size': 1024}. Best is trial 2 with value: 0.8921398317992478.
[I 2026-11-18 07:08:35,610] Trial 3 finished with value: 0.892123076809216 and parameters: {'units1': 128, 'units2': 64, 'activation': 'relu', 'alpha': 2.4586032763280077e-06, 'lr': 0.0023359635026261607, 'batch

optimisation time: 1468 s, trials: 20
untuned (trial 0) CV Top-1: 0.8896
best CV Top-1: 0.8921 | best params: {'units1': 128, 'units2': 64, 'activation': 'tanh', 'alpha': 2.9204338471814107e-05, 'lr': 0.000816845589476017, 'batch_size': 1024}


In [ ]:
best = study.best_params
mlp_tuned = run_model("MLP_ctx_tuned", "Neural network", f"Optuna best: {best}", make_mlp(best))
pd.DataFrame(results)[["model", "cv_macro_f1", "test_top1_acc", "test_macro_f1", "test_log_loss", "route_continuity", "train_time_s"]]

--- MLP_ctx_tuned


/usr/local/lib/python3.13/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (100) reached and the optimization hasn't converged yet.
  warnings.warn(


{'model': 'MLP_ctx_tuned', 'cv_macro_f1': np.float64(0.9353), 'cv_std': np.float64(0.0099), 'test_top1_acc': np.float64(0.8921), 'test_macro_f1': 0.9325, 'test_precision': 0.9331, 'test_recall': 0.9319, 'test_log_loss': 0.0846, 'route_continuity': np.float64(0.9573), 'train_time_s': 698.7, 'inference_ms_per_point': 0.0359} | total 2045s


,model,cv_macro_f1,test_top1_acc,test_macro_f1,test_log_loss,route_continuity,train_time_s
0,LogisticRegression,0.8705,0.8509,0.8698,0.2047,0.9196,5.1
1,GaussianNB,0.7986,0.8284,0.7979,0.5774,0.9065,0.4
2,DecisionTree,0.8770,0.8502,0.8824,0.1853,0.9142,20.3
3,RandomForest,0.8908,0.8493,0.8926,0.1613,0.9152,182.5
4,HistGradientBoosting,0.8865,0.8506,0.8847,0.1748,0.9139,23.8
5,KNN,0.9012,0.8492,0.9084,0.1642,0.9136,5.1
6,MLP,0.9028,0.8500,0.9095,0.1170,0.9160,138.8
7,HistGradientBoosting_ctx,0.9256,0.8898,0.9152,0.1213,0.9543,25.6
8,LogisticRegression_ctx,0.9049,0.8810,0.8961,0.1573,0.9508,5.8
9,MLP_ctx,0.9350,0.8922,0.9337,0.0810,0.9601,367.6
